## Load modules

In [1]:
!module load scipy-stack/2023b
!module list

Lmod Warning: 
-------------------------------------------------------------------------------
The following dependent module(s) are not currently loaded: ipykernel/2026a
(required by: ipython-kernel/3.11)
-------------------------------------------------------------------------------




Currently Loaded Modules:
  1) CCconfig                 16) calibre/8.6.0
  2) gentoo/2023         (S)  17) libreqda/1.1.0
  3) gcccore/.12.3       (H)  18) flexiblascore/.3.3.1         (H)
  4) gcc/12.3            (t)  19) java/17 -> java/17.0.6       (t)
  5) hwloc/2.9.1              20) r/4.4.0                      (t)
  6) ucx/1.14.1               21) rstudio-server/4.4           (t)
  7) libfabric/1.18.0         22) python/3.11 -> python/3.11.5 (t)
  8) pmix/4.2.4               23) ipython-kernel/3.11          (S)
  9) ucc/1.2.0                24) openrefine/3.9.3
 10) openmpi/4.1.5       (m)  25) mlflow/3.8.1
 11) flexiblas/3.3.1          26) tensorboard/2.20.0
 12) blis/0.9.0               27) 

In [5]:
import numpy as np
import pandas as pd
import math as math
import matplotlib.pyplot as plt
import subprocess
import os
import scipy.stats as stats
import seaborn as sns
from collections import Counter
from tqdm import tqdm
import glob
from matplotlib.lines import Line2D

In [6]:
input_datasheet = pd.read_csv('../ChrR_datasheet.csv', sep=',', index_col=0)
df = pd.read_csv('../2.Filtering_and_Statistics/statistics_df_with_log2fc_and_fdr.csv', sep=',', index_col=0)
df

,Alias,Gene,Barcode,Position,SC5314_TP0_YPD_1,SC5314_TP0_YPD_2,SC5314_TP0_YPD_3,SC5314_TP0_YPD_4,SC5314_TP3_YPD_1,SC5314_TP3_YPD_2,...,freq_SC5314_TP3_High_FLZ_4,log2_fold_change_ypd,log2_fold_change_low_flz,log2_fold_change_high_flz,p_value_ypd,p_value_low_flz,p_value_high_flz,adj_p_value_ypd,adj_p_value_low_flz,adj_p_value_high_flz
Number,,,,,,,,,,,,,,,,,,,,,
1,CR07390CA_242,CR07390C_A,TGCGCACAATTTCCTGCACA,1605752.0,16777.0,25374.0,37679.0,24299.0,53161.0,47017.0,...,0.001372,0.917425,-0.286737,0.139177,0.000353,0.179176,0.431535,0.000826,0.207847,0.468376
2,CR07390CA_276_revcom,CR07390C_A,GAGTATAGTGATCCATGTGC,1605740.0,1630.0,1138.0,1558.0,1063.0,212.0,155.0,...,0.000002,-2.794206,-2.978897,-3.962058,0.000060,0.000435,0.000251,0.000231,0.001118,0.000687
3,CR07390CA_239_revcom,CR07390C_A,GCTATAACGTTACTAGTAGT,1605777.0,2233.0,2024.0,2230.0,1768.0,81.0,597.0,...,0.000167,-2.175645,1.149370,0.146875,0.000382,0.000003,0.758596,0.000876,0.000060,0.785456
6,CR07390CA_70,CR07390C_A,CTCTTTTGCTCTCATTTTTA,1605924.0,3537.0,2787.0,4190.0,2699.0,3875.0,3517.0,...,0.000138,0.260456,-0.045774,-0.185326,0.094036,0.873215,0.354892,0.111381,0.883694,0.391744
7,CR10440WA_195_revcom,CR10440W_A,AGGGGTTTAAGCATACTATC,2222654.0,2499.0,1995.0,2960.0,2248.0,2791.0,2333.0,...,0.000080,-0.110935,-1.154587,-0.951955,0.459915,0.001511,0.003591,0.490137,0.002973,0.006109
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5692,non-targeting52,non-targeting52,ACGATTTTGACCGAGCCCCA,NaN,5104.0,5851.0,7459.0,5822.0,8716.0,6515.0,...,0.000112,0.323465,-1.537049,-1.393856,0.021533,0.000126,0.000141,0.028900,0.000461,0.000462
5695,non-targeting55,non-targeting55,AGTGTCCCGTCATGCTCCAG,NaN,7059.0,5164.0,5634.0,4665.0,1735.0,1252.0,...,0.000006,-1.608557,-2.559154,-3.977842,0.001078,0.000501,0.000106,0.002046,0.001247,0.000371
5698,non-targeting58,non-targeting58,GACCTCTTTCGACTAGGCCA,NaN,6683.0,7238.0,9541.0,7239.0,12218.0,11650.0,...,0.000266,0.470098,-0.425868,-0.348164,0.000492,0.084308,0.128982,0.001064,0.104150,0.156286


## Plotting

Highlight the FLZ-unique hits (ie. hits are only considered if they were NOT also a hit in the YPD-only condition) and add the non-targeting sgRNA confidence intervals.

In [9]:
#Directories
output_dir = "./volcano_plots"
os.makedirs(output_dir, exist_ok=True)

#Thresholds. Absolute logfc thresholds are optional- here we will use the thresholds based on the non-targeting sgRNA confidence interval, so this is left at 0
fdr_thresh = 0.05
logfc_thresh = 0

#List of comparisons with column names in the .csv: (log2FC column, adjusted p-value column)
comparisons = [
    ("log2_fold_change_ypd",      "adj_p_value_ypd"),
    ("log2_fold_change_low_flz",  "adj_p_value_low_flz"),
    ("log2_fold_change_high_flz", "adj_p_value_high_flz"),
]

#Columns
high_flz_logfc_col = "log2_fold_change_high_flz"
high_flz_fdr_col   = "adj_p_value_high_flz"
low_flz_logfc_col  = "log2_fold_change_low_flz"
low_flz_fdr_col    = "adj_p_value_low_flz"
ypd_logfc_col      = "log2_fold_change_ypd"
ypd_fdr_col        = "adj_p_value_ypd"

do_flz_only_highlight = True

#Filter non-targeting guides correctly (safer to avoid NaNs)
non_targeting_df = df[df['Alias'].str.startswith('non-targeting', na=False)]

#Compute means and 95% CI for each comparison (based on non-targeting logFC distribution)
#This is basically a normal distribution-based CI on the non-targeting logFC values (ie. contains ~95% of individual non-targeting sgRNA log2FC values)
ci_dict = {}
for logfc_col, _ in comparisons:
    nt_vals = non_targeting_df[logfc_col].dropna()
    n_nt = nt_vals.shape[0]
    mean_fc = nt_vals.mean()
    std_fc = nt_vals.std()
    ci_lower = mean_fc - 1.96 * std_fc
    ci_upper = mean_fc + 1.96 * std_fc
    ci_dict[logfc_col] = (mean_fc, ci_lower, ci_upper)
    print(f"{logfc_col}: n_non_targeting={n_nt}, mean={mean_fc:.4f}, 95% CI = [{ci_lower:.4f}, {ci_upper:.4f}]")

#Build YPD direction dictionary
ypd_mean, ypd_ci_lower, ypd_ci_upper = ci_dict[ypd_logfc_col]
ypd_outside_ci = (df[ypd_logfc_col] < ypd_ci_lower) | (df[ypd_logfc_col] > ypd_ci_upper)

ypd_sig_outside_ci = (
    (df[ypd_fdr_col] < fdr_thresh) &
    (df[ypd_logfc_col].abs() > logfc_thresh) &
    ypd_outside_ci
)

#Build a lookup of YPD-significant hits and their direction (+1 enriched, -1 depleted) so that FLZ hits in the same "direction" as YPD can be flagged as shared rather than FLZ-unique
tmp = df.loc[ypd_sig_outside_ci, ['Alias', ypd_logfc_col]].dropna()
ypd_direction = dict(zip(tmp['Alias'], np.sign(tmp[ypd_logfc_col])))

#Legend handles
legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='red', markersize=5, label='FLZ-Unique Significant'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='black',  markersize=5, label='Significant'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='grey', markersize=5, label='Not significant')
]

#Plotting
for logfc_col, fdr_col in comparisons:
    df['minus_log10_FDR'] = -np.log10(df[fdr_col].replace(0, np.nextafter(0, 1)))

    #CI for Tthe current plot's logFC distribution (non-targeting)
    mean_fc, ci_lower, ci_upper = ci_dict[logfc_col]

    #Treat anything inside CI as NOT significant
    outside_ci = (df[logfc_col] < ci_lower) | (df[logfc_col] > ci_upper)

    #"Significant" for the current plot means they pass the FDR/logFC thresholds AND is outside the NTC CI
    sig_this = (df[fdr_col] < fdr_thresh) & (df[logfc_col].abs() > logfc_thresh) & outside_ci
    df['significant'] = sig_this

    #FLZ-unique highlighting ONLY on the corresponding FLZ plot, and ONLY exclude YPD hits if they score in the SAME DIRECTION (+ or -)
    is_flz_plot = logfc_col in {low_flz_logfc_col, high_flz_logfc_col}

    if do_flz_only_highlight and is_flz_plot:
        #Same-direction YPD exclusion mask
        ypd_dir_series = df['Alias'].map(ypd_direction)  # NaN for non-YPD hits
        same_direction_as_ypd = (ypd_dir_series.notna()) & (np.sign(df[logfc_col]) == ypd_dir_series)
        flz_unique = sig_this & (~same_direction_as_ypd)
    else:
        flz_unique = np.zeros(len(df), dtype=bool)

    #Colours: red (FLZ-unique) > black (significant) > grey
    colors = np.where(flz_unique, 'red', np.where(sig_this, 'black', 'grey'))
    plt.figure(figsize=(5, 5))
    plt.scatter(
        df[logfc_col], df['minus_log10_FDR'],
        c=colors,
        s=13, edgecolors='none'
    )

    # Fixed y-axis limits
    y_min = 0
    y_max = 8
    plt.ylim(y_min, y_max)

    # Fixed x-axis limits (adjust as needed)
    x_min = -9
    x_max = 9
    plt.xlim(x_min, x_max)

    #FDR threshold line
    plt.axhline(-np.log10(fdr_thresh), color='black', ls='--', lw=1)

    #Add shaded confidence interval for non-targeting guides
    plt.fill_betweenx([y_min, y_max], ci_lower, ci_upper, color='gray', alpha=0.2, edgecolor='none')
    plt.xlabel('log2 Fold Change', fontsize=10, fontweight="bold")
    plt.ylabel('-log10 FDR', fontsize=10, fontweight="bold")
    ax = plt.gca()
    ax.tick_params(axis='both', labelsize=10)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')

    #Add legend only on the FLZ plots (wouldn't make sense on the YPD plots)
    if is_flz_plot and do_flz_only_highlight:
        leg = ax.legend(
            handles=legend_handles,
            frameon=False,
            fontsize=8,
            loc='upper left',
            handletextpad=0.4,
            borderpad=0.2,
            labelspacing=0.3,
        )
        for text in leg.get_texts():
            text.set_fontweight('bold')

    #Add FLZ condition label in top-right
    if logfc_col == high_flz_logfc_col:
        ax.text(
            0.98, 0.98,
            '64μg/mL FLZ',
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=10,
            fontweight='bold'
        )

    elif logfc_col == low_flz_logfc_col:
        ax.text(
            0.98, 0.98,
            '1μg/mL FLZ',
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=10,
            fontweight='bold'
        )

    plt.tight_layout()
    plot_path = os.path.join(output_dir, f"volcano_{logfc_col}_with_CI.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()

log2_fold_change_ypd: n_non_targeting=30, mean=0.1686, 95% CI = [-1.3742, 1.7114]
log2_fold_change_low_flz: n_non_targeting=30, mean=-0.9266, 95% CI = [-2.8800, 1.0267]
log2_fold_change_high_flz: n_non_targeting=30, mean=-0.7291, 95% CI = [-3.0950, 1.6368]


In [10]:
#Summary table of the sgRNAs enriched/depleted beyond the non-targeting CI in FLZ conditions
records = []

for logfc_col, fdr_col in [
    (low_flz_logfc_col,  "adj_p_value_low_flz"),
    (high_flz_logfc_col, "adj_p_value_high_flz"),
]:
    mean_fc, ci_lower, ci_upper = ci_dict[logfc_col]
    outside_ci = (df[logfc_col] < ci_lower) | (df[logfc_col] > ci_upper)
    sig_mask   = (
        (df[fdr_col] < fdr_thresh) &
        (df[logfc_col].abs() > logfc_thresh) &
        outside_ci
    )

    #Direction split
    enriched_mask  = sig_mask & (df[logfc_col] > 0)
    depleted_mask  = sig_mask & (df[logfc_col] < 0)

    #FLZ-unique: same-direction YPD exclusion
    ypd_dir_series     = df['Alias'].map(ypd_direction)
    same_dir_as_ypd    = ypd_dir_series.notna() & (np.sign(df[logfc_col]) == ypd_dir_series)
    flz_unique_mask    = sig_mask & ~same_dir_as_ypd
    flz_unique_enr  = flz_unique_mask & (df[logfc_col] > 0)
    flz_unique_dep  = flz_unique_mask & (df[logfc_col] < 0)
    shared_enr      = enriched_mask   & same_dir_as_ypd
    shared_dep      = depleted_mask   & same_dir_as_ypd

    label = "Low FLZ" if logfc_col == low_flz_logfc_col else "High FLZ"

    records.append({
        "Condition":              label,
        "Total significant":      int(sig_mask.sum()),
        "Enriched (total)":       int(enriched_mask.sum()),
        "Depleted (total)":       int(depleted_mask.sum()),
        "FLZ-unique enriched":    int(flz_unique_enr.sum()),
        "FLZ-unique depleted":    int(flz_unique_dep.sum()),
        "Shared enriched (YPD)":  int(shared_enr.sum()),
        "Shared depleted (YPD)":  int(shared_dep.sum()),
        "CI lower":               round(ci_lower, 4),
        "CI upper":               round(ci_upper, 4),
    })

summary_df = pd.DataFrame(records)

#Save CSV and print table
csv_path = os.path.join(output_dir, "flz_sgrna_summary.csv")
summary_df.to_csv(csv_path, index=False)
print(summary_df.to_string(index=False))

Condition  Total significant  Enriched (total)  Depleted (total)  FLZ-unique enriched  FLZ-unique depleted  Shared enriched (YPD)  Shared depleted (YPD)  CI lower  CI upper
  Low FLZ                329               106               223                  105                   88                      1                    135    -2.880    1.0267
 High FLZ                281                72               209                   72                   43                      0                    166    -3.095    1.6368
